> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


## Production imports (active)

```python
from mrta.retrieval import Embedder, VectorStore
from mrta.core.llm import LLMClient
from mrta.core.rag_pipeline import rag_query
from mrta.prompts import load_prompt
from mrta.observability.logging import StructuredLogger
```

See [`../../production-ready.md`](../../production-ready.md) — Phase 04.

# Phase 4 — End-to-End RAG Pipeline (with Ollama)

**Goal:** Glue together everything from Phases 1–3 into a single function that takes a question and a corpus, retrieves the most relevant chunks, and returns a grounded answer with page citations — running entirely on a local Ollama model.

This is the **first real milestone**: the moment the project becomes useful to a human.

```
Question
   |
   v
Embed question  ->  FAISS search  ->  top-k chunks
                                          |
                                          v
                              build prompt (system + context + question)
                                          |
                                          v
                                       Ollama LLM
                                          |
                                          v
                            Answer + parsed [page X] citations
```


## 4.1 — Reload the vector store


In [2]:
import os, sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from mrta.retrieval import Embedder, VectorStore

embedder = Embedder()
store = VectorStore.load("data/vector_store/aiayn", embedder)
print("Loaded vector store:", store._index.ntotal, "vectors")

Loaded vector store: 25 vectors


## 4.2 — The prompt template

A good RAG prompt has four ingredients:

1. **Role / behavior** — what kind of answers you want.
2. **Grounding rule** — tell the model to refuse if context is insufficient.
3. **Context block** — the retrieved chunks, each labeled with its source.
4. **Citation format** — explicit format the model must use.

We make this a Jinja2 template so it lives next to other prompts in `src/mrta/prompts/`.


In [3]:
from mrta.prompts import load_prompt

# Demo: render the template with a single placeholder chunk
demo_chunk_dict = {"source": "x.pdf", "page": 1, "text": "demo"}

class _ChunkProxy:
    """Minimal proxy so the Jinja2 template can access .source / .page / .text attributes."""
    def __init__(self, d):
        self.source = d["source"]; self.page = d["page"]; self.text = d["text"]

print(load_prompt("rag", chunks=[_ChunkProxy(demo_chunk_dict)], question="?")[:400])

You are a careful research assistant. Answer the user's question using ONLY the context below.
If the context is insufficient, say "I could not find this in the provided documents."

For every factual claim, cite the source page in square brackets like [page 3] or [page 3, 5].
Be precise. Do not invent details, equations, or numbers.

--- CONTEXT ---

[chunk 1 | x.pdf | page 1]
demo


--- QUESTION


## 4.3 — The LLM client wrapper

`src/mrta/core/llm.py` exposes a single `chat(messages, **opts)` function so the rest of the code does not care which provider is behind it.


In [9]:
from mrta.core.llm import LLMClient

llm = LLMClient(model="llama3.2:latest")  # reads settings.llm_provider and settings.ollama_llm_model
print("LLM client ready:", llm.model)

LLM client ready: llama3.2:latest


In [10]:
llm.model

'llama3.2:latest'

## 4.4 — The RAG pipeline as one function


In [6]:
from mrta.core.rag_pipeline import rag_query

## 4.5 — Try it


In [11]:
out = rag_query("What is multi-head attention and why is it used?", vector_store=store, llm=llm, top_k=5)

print("Q: What is multi-head attention and why is it used?")
print("\nA:", out["answer"])
print("\nSources (pages):", [c.page for c in out["sources"]])
print(f"Latency: {out['latency_s']:.1f}s")

Q: What is multi-head attention and why is it used?

A: Multi-head attention is a technique used in the Transformer model, which allows the model to jointly attend to information from different representation subspaces at different positions. This is achieved by applying multiple attention heads in parallel, where each head performs the same attention operation but on different subsets of the input data.

The use of multi-head attention is motivated by the fact that a single attention head can inhibit the ability to attend to information from different representation subspaces. By using multiple heads, the model can capture this information and learn more robust representations.

In the Transformer model, multi-head attention is used in three different ways:

1. In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder.
2. The encoder contains self-attention layers, where all of the keys, 

## 4.6 — A few more questions


In [12]:
questions = [
    "What dataset and tokenization were used for English-to-German training?",
    "How does scaled dot-product attention scale with sequence length?",
    "Why use sinusoidal positional encodings instead of learned ones?",
    "What were the BLEU scores reported on WMT 2014 English-to-French?",
]

for q in questions:
    out = rag_query(q, vector_store=store, llm=llm, top_k=5)
    print(f"Q: {q}")
    print(f"A: {out['answer'][:280]}{'...' if len(out['answer']) > 280 else ''}")
    print(f"   sources={[c.page for c in out['sources']]}  latency={out['latency_s']:.1f}s\n")

Q: What dataset and tokenization were used for English-to-German training?
A: The dataset used for the English-to-German training was the Penn Treebank, specifically the Wall Street Journal (WSJ) portion. The vocabulary size used for this setting was 16K tokens.

However, I could not find information on the specific tokenization scheme used in the provided...
   sources=[9, 8, 7, 11, 10]  latency=1.9s

Q: How does scaled dot-product attention scale with sequence length?
A: According to the provided context, scaled dot-product attention scales with sequence length due to the growth of the dot products between queries and keys. 

The text states: "To illustrate why the dot products get large, assume that the components of q and k are independent rand...
   sources=[4, 7, 10, 6, 5]  latency=2.9s

Q: Why use sinusoidal positional encodings instead of learned ones?
A: According to the text, the authors chose to use sinusoidal positional encodings instead of learned ones because they hypothe

## 4.7 — Observability: log every run

Senior-engineer move. Every answer becomes a structured JSON line we can inspect, replay, and feed into evaluation.


In [13]:
from mrta.observability.logging import StructuredLogger

logger = StructuredLogger()

# Re-run the first question and log it.
out = rag_query("What is multi-head attention and why is it used?", vector_store=store, llm=llm)
logger.log_run(
    question="What is multi-head attention and why is it used?",
    answer=out["answer"],
    sources=out["sources"],
    latency_s=out["latency_s"],
)
from mrta.core.config import settings
print(f"Logged 1 run → {settings.log_file}")

Logged 1 run → data/logs/runs.jsonl


## 4.8 — Failure modes to watch for

| Symptom                              | Likely cause                                                 | Fix                                                  |
|--------------------------------------|--------------------------------------------------------------|------------------------------------------------------|
| "I could not find this..." too often | k too low, or chunk size too small                           | Raise `k`, or rerun chunking with `chunk_size=900`.  |
| Confident but wrong answers          | LLM ignoring context                                         | Strengthen system prompt; lower temperature; add a re-ask "is this in the context?" pass. |
| Cites the wrong page                 | Citation parser sees a page number inside the answer text    | Make the model emit citations only as `[page N]`, and post-validate against retrieved pages. |
| Cuts mid-sentence                    | LLM context too small for chunks + question + answer         | Reduce `k`, or summarize chunks before feeding.       |

We will quantify all of these in Notebook 09 with Ragas.


## What you learned

- The standard RAG flow: embed → retrieve → prompt → answer + parse citations.
- A Jinja2 prompt template you can iterate on.
- An LLM client wrapper that returns latency + token counts.
- Why structured JSONL logging is a senior-engineer move.
- Common RAG failure modes and what to tune.

## Exercises

1. Add a `reranker` step: re-score the top-20 chunks with a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) and keep the top-5. Measure latency vs quality.
2. Add a "no context" guard: if the highest retrieval score is below a threshold (say 0.3), short-circuit and answer "I am not confident I can answer from these documents."
3. Make the citation parser also validate that every cited page was actually retrieved.

## Next: [Phase 5 — FastAPI backend](./2026-05-25-phase05-fastapi-backend.ipynb)


# Solutions

## Exercise 1 — Reranker

Concept: Bi-encoder (FAISS) retrieval is fast but coarse — it compares query and chunk embeddings independently. A cross-encoder reads the query and chunk together, giving a much more accurate relevance score. The tradeoff is speed: a cross-encoder scores one pair at a time, so you use it only on the top-20 FAISS candidates, not on the whole corpus.


query ──── FAISS ──── top-20 ──── CrossEncoder ──── top-5 ──── LLM
           (fast,              (slow, accurate,
           coarse)             runs on 20 pairs)
Reranker is already implemented in reranker.py and rag_query already accepts it:


import time
from mrta.retrieval.reranker import Reranker

reranker = Reranker("cross-encoder/ms-marco-MiniLM-L-6-v2")
q = "What is multi-head attention and why is it used?"

# Without reranker
t0 = time.time()
out_base = rag_query(q, vector_store=store, llm=llm, top_k=5)
t_base = time.time() - t0

# With reranker: fetch 20, rerank to 5
t0 = time.time()
out_rerank = rag_query(q, vector_store=store, llm=llm, top_k=20,
                       reranker=reranker, rerank_top_n=5)
t_rerank = time.time() - t0

print(f"Base    — {t_base:.1f}s  sources: {[c.page for c in out_base['sources']]}")
print(f"Reranked — {t_rerank:.1f}s  sources: {[c.page for c in out_rerank['sources']]}")
print("\n--- Base answer ---\n",    out_base["answer"][:300])
print("\n--- Reranked answer ---\n", out_rerank["answer"][:300])
What to look for: the reranked answer usually has tighter citations (fewer pages, more relevant ones). The latency hit is the cross-encoder scoring 20 pairs — typically +0.5–2s on CPU.

## Exercise 2 — No-context guard

Concept: FAISS always returns something even when nothing is relevant. If a user asks "What is the weather today?" against a paper corpus, FAISS will still return 5 chunks with low similarity scores. The guard detects this and short-circuits before the LLM wastes time generating a hallucinated answer.

VectorStore.search doesn't expose scores, so add a thin helper in the notebook:


import numpy as np

def search_with_scores(store, query: str, k: int = 5):
    """Returns (chunks, scores) — scores are cosine similarities in [0, 1]."""
    q = store._embedder.embed([query])
    scores, idx = store._ensure_index().search(q, k)
    chunks = [store._chunks[i] for i in idx[0] if 0 <= i < len(store._chunks)]
    return chunks, scores[0].tolist()

CONFIDENCE_THRESHOLD = 0.3

def rag_query_guarded(question, store, llm, top_k=5, threshold=CONFIDENCE_THRESHOLD):
    chunks, scores = search_with_scores(store, question, k=top_k)
    print(f"  top score: {scores[0]:.3f}")
    if not chunks or scores[0] < threshold:
        return {
            "answer": "I am not confident I can answer from these documents.",
            "sources": [],
            "latency_s": 0.0,
            "guarded": True,
        }
    return rag_query(question, vector_store=store, llm=llm, top_k=top_k)

# Test with a relevant question and an off-topic one
rag_query_guarded("What is multi-head attention?", store, llm)
rag_query_guarded("What is the weather in Paris today?", store, llm)
The second call should hit the guard and return the short-circuit answer without calling the LLM.

## Exercise 3 — Citation validation

Concept: The LLM is instructed to cite [page N] but nothing enforces it. It can cite a page that was never retrieved (hallucinated citation) or mis-number a page. Validating against the actual sources list catches both cases.


import re

def validate_citations(answer: str, sources: list) -> dict:
    retrieved_pages = {c.page for c in sources}
    cited = [int(m.group(1)) for m in re.finditer(r"\[page (\d+)", answer, re.IGNORECASE)]

    hallucinated = [p for p in cited if p not in retrieved_pages]
    valid        = [p for p in cited if p in retrieved_pages]

    return {
        "cited"       : cited,
        "retrieved"   : sorted(retrieved_pages),
        "valid"       : valid,
        "hallucinated": hallucinated,
        "clean"       : len(hallucinated) == 0,
    }

# Run on the earlier rag_query output
out = rag_query("What is multi-head attention?", vector_store=store, llm=llm, top_k=5)
report = validate_citations(out["answer"], out["sources"])

print("Answer  :", out["answer"][:200])
print("Cited   :", report["cited"])
print("Retrieved pages:", report["retrieved"])
print("Hallucinated:", report["hallucinated"])
print("Clean   :", report["clean"])
Note that citation_correctness in mrta/evaluation/metrics.py already implements exactly this logic — this exercise is essentially re-discovering it from scratch so you understand why it exists.